# 📊 Instagram Content Performance Predictive Analysis
## Turning Data Into a Brand Content Strategy

---

### 🎯 Project Goal
Build a machine learning pipeline to identify patterns in Instagram post performance,
enabling brands to make **data-driven content decisions** that maximize engagement and follower growth.

---

### 📋 STAR Method Overview

| | |
|---|---|
| **Situation** | Brands invest heavily in Instagram content without knowing which posts will perform well |
| **Task** | Analyze 29,999 Instagram posts to uncover high-performance content patterns |
| **Action** | Built a full EDA + supervised ML pipeline across 9 classifiers and 3 regressors |
| **Result** | Actionable content strategy insights + reusable prediction framework |

---

### 📦 Dataset
- **Source:** Instagram Analytics Case Study — 29,999 posts
- **Features:** Media type, content category, caption length, hashtags, posting time, traffic source
- **Targets:** Engagement rate, follower growth, high/low engagement classification

---
**Author:** Samuel Madumere
**Date:** May 2026
**Tools:** Python · Scikit-learn · Pandas · Matplotlib · Seaborn


## 1. 📦 Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches
import seaborn as sns
import warnings, os, json
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, ExtraTreesClassifier,
                               RandomForestRegressor, GradientBoostingRegressor)
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, roc_curve, auc,
                              mean_squared_error, mean_absolute_error, r2_score)

# ── Dark theme constants ──
BG      = "#0D0D0D";  SURFACE = "#1A1A2E"; CARD = "#16213E"
ACCENT1 = "#00D4FF";  ACCENT2 = "#7B2FBE"; ACCENT3 = "#FF6B35"
ACCENT4 = "#00FF88";  ACCENT5 = "#FF3CAC"; TEXT = "#E0E0E0"; GRID = "#2A2A4A"
PALETTE = [ACCENT1, ACCENT2, ACCENT3, ACCENT4, ACCENT5, "#FFD700", "#FF4560", "#00E396"]

plt.rcParams.update({
    'figure.facecolor': BG,  'axes.facecolor': SURFACE, 'axes.edgecolor': GRID,
    'axes.labelcolor':  TEXT, 'xtick.color': TEXT,      'ytick.color': TEXT,
    'text.color': TEXT,       'grid.color': GRID,        'grid.alpha': 0.4,
    'axes.grid': True,        'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
})

VIZ_DIR = "visualizations"
os.makedirs(VIZ_DIR, exist_ok=True)

def save_fig(name):
    path = f"{VIZ_DIR}/{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()
    print(f"  ✓ Saved {path}")

print("✅ All imports loaded. Dark theme set.")


## 2. 📂 Data Loading & Overview

We use the **cleaned and feature-engineered** version of the dataset which includes
pre-computed engagement metrics and temporal features extracted from upload dates.


In [ ]:
df = pd.read_csv("/Users/apple/Desktop/Instagram Analytics Datasets/Full Dataset/Intagram_CaseStudy_Cleaned_sql.csv")

print(f"Dataset Shape:  {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumn Names:\n{list(df.columns)}")
print(f"\nData Types:\n{df.dtypes}")


In [ ]:
print("\n📊 Statistical Summary:")
df.describe().T.round(2)


In [ ]:
print("\n🔍 Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "✅ No missing values found.")

print("\n📋 Categorical Value Distributions:")
for col in ['media_type', 'traffic_source', 'content_category']:
    print(f"\n{col}:")
    print(df[col].value_counts().to_string())


## 3. ⚙️ Feature Engineering

### Target Variable: High Engagement
We define **high engagement** as posts with an engagement rate **at or above the median**.
This creates a balanced 50/50 binary classification target.

> **Engagement Rate** = (Likes + Comments + Shares + Saves) / Reach × 100


In [ ]:
median_er = df['engagement_rate_calc'].median()
df['high_engagement'] = (df['engagement_rate_calc'] >= median_er).astype(int)

print(f"Median Engagement Rate:  {median_er:.4f}%")
print(f"High Engagement Posts:   {df['high_engagement'].sum():,} ({df['high_engagement'].mean()*100:.1f}%)")
print(f"Low Engagement Posts:    {(1-df['high_engagement']).sum():,} ({(1-df['high_engagement']).mean()*100:.1f}%)")

# Engagement tier for EDA
df['engagement_tier'] = pd.cut(df['engagement_rate_calc'],
    bins=[0, df['engagement_rate_calc'].quantile(0.33),
             df['engagement_rate_calc'].quantile(0.67), float('inf')],
    labels=['Low', 'Medium', 'High'])

df[['engagement_rate_calc', 'high_engagement', 'engagement_tier']].head(10)


## 4. 🎨 Exploratory Data Analysis (EDA)

We explore 10 key dimensions of the data to surface actionable content strategy insights.


### 4.1 Target Distribution — How Balanced Is Our Dataset?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)

labels = ['Low Engagement', 'High Engagement']
counts = df['high_engagement'].value_counts().sort_index()
bars = axes[0].bar(labels, counts.values, color=[ACCENT2, ACCENT1], edgecolor=BG, linewidth=1.5, width=0.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom',
                 color=TEXT, fontsize=11, fontweight='bold')
axes[0].set_title('Post Engagement Distribution', color=TEXT, fontsize=14, fontweight='bold', pad=12)
axes[0].set_ylabel('Number of Posts', color=TEXT)

wedges, texts, autotexts = axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
    colors=[ACCENT2, ACCENT1], startangle=90, textprops={'color': TEXT},
    wedgeprops={'edgecolor': BG, 'linewidth': 2})
for at in autotexts:
    at.set_color(BG); at.set_fontweight('bold')
axes[1].set_title('Engagement Split', color=TEXT, fontsize=14, fontweight='bold', pad=12)
save_fig("01_target_distribution")


### 4.2 Engagement Rate by Media Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor(BG)

media_eng = df.groupby('media_type')['engagement_rate_calc'].mean().sort_values(ascending=False)
colors = PALETTE[:len(media_eng)]
bars = axes[0].bar(media_eng.index, media_eng.values, color=colors, edgecolor=BG, linewidth=1.2)
for bar, val in zip(bars, media_eng.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.2f}%', ha='center', va='bottom', color=TEXT, fontsize=10, fontweight='bold')
axes[0].set_title('Avg Engagement Rate by Media Type', color=TEXT, fontsize=13, fontweight='bold')
axes[0].set_ylabel('Avg Engagement Rate (%)', color=TEXT)
axes[0].set_xlabel('Media Type', color=TEXT)

df.boxplot(column='engagement_rate_calc', by='media_type', ax=axes[1], patch_artist=True)
axes[1].set_title('Engagement Rate Distribution by Media Type', color=TEXT, fontsize=13, fontweight='bold')
axes[1].set_xlabel('Media Type', color=TEXT)
axes[1].set_ylabel('Engagement Rate (%)', color=TEXT)
plt.suptitle('')
for patch, color in zip(axes[1].findobj(matplotlib.patches.PathPatch), PALETTE):
    patch.set_facecolor(color); patch.set_alpha(0.7)
save_fig("02_engagement_by_media_type")

print(f"\n🏆 Best Media Type: {media_eng.idxmax()} ({media_eng.max():.2f}% avg engagement rate)")


### 4.3 Engagement Rate by Content Category

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
fig.patch.set_facecolor(BG)
cat_eng = df.groupby('content_category')['engagement_rate_calc'].mean().sort_values(ascending=False)
bars = ax.bar(cat_eng.index, cat_eng.values, color=PALETTE[:len(cat_eng)], edgecolor=BG, linewidth=1.2)
for bar, val in zip(bars, cat_eng.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{val:.2f}%', ha='center', va='bottom', color=TEXT, fontsize=9, fontweight='bold')
ax.set_title('Average Engagement Rate by Content Category', color=TEXT, fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Avg Engagement Rate (%)', color=TEXT)
ax.set_xlabel('Content Category', color=TEXT)
plt.xticks(rotation=30, ha='right')
save_fig("03_engagement_by_category")

print(f"\n🏆 Best Category: {cat_eng.idxmax()} ({cat_eng.max():.2f}% avg engagement rate)")


### 4.4 Engagement Rate by Day of Week

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_eng   = df.groupby('Day name')['engagement_rate_calc'].mean().reindex(day_order)

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor(BG)
ax.plot(day_order, day_eng.values, color=ACCENT1, linewidth=2.5, marker='o',
        markersize=9, markerfacecolor=ACCENT3, markeredgecolor=BG, markeredgewidth=1.5)
ax.fill_between(day_order, day_eng.values, alpha=0.15, color=ACCENT1)
for day, val in zip(day_order, day_eng.values):
    ax.annotate(f'{val:.2f}%', (day, val), textcoords='offset points',
                xytext=(0, 10), ha='center', color=TEXT, fontsize=9, fontweight='bold')
ax.set_title('Average Engagement Rate by Day of Week', color=TEXT, fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Avg Engagement Rate (%)', color=TEXT)
ax.set_xlabel('Day of Week', color=TEXT)
save_fig("04_engagement_by_day")

print(f"\n🏆 Best Day to Post: {day_eng.idxmax()} ({day_eng.max():.2f}% avg engagement rate)")


### 4.5 Hashtag Count & Caption Length vs Engagement

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor(BG)

hashtag_bins = pd.cut(df['hashtags_count'], bins=[0,5,10,15,20,30], labels=['1-5','6-10','11-15','16-20','21-30'])
hash_eng = df.groupby(hashtag_bins)['engagement_rate_calc'].mean()
bars = axes[0].bar(hash_eng.index.astype(str), hash_eng.values, color=PALETTE[:5], edgecolor=BG, linewidth=1.2)
for bar, val in zip(bars, hash_eng.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}%', ha='center', va='bottom', color=TEXT, fontsize=10, fontweight='bold')
axes[0].set_title('Engagement Rate by Hashtag Count', color=TEXT, fontsize=13, fontweight='bold')
axes[0].set_xlabel('Hashtag Count Range', color=TEXT)
axes[0].set_ylabel('Avg Engagement Rate (%)', color=TEXT)

caption_bins = pd.cut(df['caption_length'], bins=[0,50,150,300,500,2200],
                      labels=['Micro\n<50','Short\n50-150','Medium\n150-300','Long\n300-500','Epic\n500+'])
cap_eng = df.groupby(caption_bins)['engagement_rate_calc'].mean()
bars2 = axes[1].bar(cap_eng.index.astype(str), cap_eng.values, color=PALETTE[:5], edgecolor=BG, linewidth=1.2)
for bar, val in zip(bars2, cap_eng.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}%', ha='center', va='bottom', color=TEXT, fontsize=10, fontweight='bold')
axes[1].set_title('Engagement Rate by Caption Length', color=TEXT, fontsize=13, fontweight='bold')
axes[1].set_xlabel('Caption Length (characters)', color=TEXT)
axes[1].set_ylabel('Avg Engagement Rate (%)', color=TEXT)
save_fig("08_hashtag_caption_analysis")


### 4.6 Correlation Heatmap

In [ ]:
numeric_cols = ['likes', 'comments', 'shares', 'saves', 'reach', 'impressions',
                'caption_length', 'hashtags_count', 'followers_gained',
                'engagement_rate_calc', 'saves_rates', 'share_rate', 'Month']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
fig.patch.set_facecolor(BG)
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
            annot=True, fmt='.2f', square=True, linewidths=0.5,
            linecolor=BG, ax=ax, annot_kws={'size': 8, 'color': TEXT},
            cbar_kws={'shrink': 0.7})
ax.set_title('Correlation Heatmap — Instagram Metrics', color=TEXT, fontsize=14, fontweight='bold', pad=12)
ax.set_facecolor(SURFACE)
save_fig("07_correlation_heatmap")


### 4.7 Traffic Source Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor(BG)
ts_eng = df.groupby('traffic_source')['engagement_rate_calc'].mean().sort_values(ascending=False)
ts_cnt = df['traffic_source'].value_counts()

bars = axes[0].barh(ts_eng.index, ts_eng.values, color=PALETTE[:len(ts_eng)], edgecolor=BG, linewidth=1.2)
for bar, val in zip(bars, ts_eng.values):
    axes[0].text(val + 0.02, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}%', va='center', color=TEXT, fontsize=10, fontweight='bold')
axes[0].set_title('Avg Engagement by Traffic Source', color=TEXT, fontsize=13, fontweight='bold')
axes[0].set_xlabel('Avg Engagement Rate (%)', color=TEXT)

wedges, texts, autotexts = axes[1].pie(ts_cnt.values, labels=ts_cnt.index,
    autopct='%1.1f%%', colors=PALETTE[:len(ts_cnt)], startangle=90,
    textprops={'color': TEXT}, wedgeprops={'edgecolor': BG, 'linewidth': 2})
for at in autotexts:
    at.set_color(BG); at.set_fontweight('bold')
axes[1].set_title('Traffic Source Distribution', color=TEXT, fontsize=13, fontweight='bold')
save_fig("09_traffic_source_analysis")


### 4.8 Monthly Engagement Trend

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor(BG)
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_eng = df.groupby('Month')['engagement_rate_calc'].mean().reindex(range(1,13))
valid_months = [m for m, v in zip(month_names, month_eng.values) if not np.isnan(v)]
valid_vals   = [v for v in month_eng.values if not np.isnan(v)]
ax.bar(valid_months, valid_vals, color=PALETTE[:len(valid_months)], edgecolor=BG, linewidth=1.2, width=0.6)
for i, val in enumerate(valid_vals):
    ax.text(i, val + 0.03, f'{val:.2f}%', ha='center', va='bottom', color=TEXT, fontsize=9, fontweight='bold')
ax.plot(valid_months, valid_vals, color=ACCENT1, linewidth=2, marker='D',
        markersize=6, markerfacecolor=ACCENT5, markeredgecolor=BG)
ax.set_title('Average Engagement Rate by Month', color=TEXT, fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Avg Engagement Rate (%)', color=TEXT)
save_fig("10_monthly_engagement_trend")


### 4.9 Followers Gained by Content Category

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
fig.patch.set_facecolor(BG)
cat_fol = df.groupby('content_category')['followers_gained'].mean().sort_values(ascending=False)
bars = ax.bar(cat_fol.index, cat_fol.values, color=PALETTE[:len(cat_fol)], edgecolor=BG, linewidth=1.2)
for bar, val in zip(bars, cat_fol.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f}', ha='center', va='bottom', color=TEXT, fontsize=9, fontweight='bold')
ax.set_title('Average Followers Gained by Content Category', color=TEXT, fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Avg Followers Gained', color=TEXT)
ax.set_xlabel('Content Category', color=TEXT)
plt.xticks(rotation=30, ha='right')
save_fig("06_followers_by_category")

cat_fol_top = cat_fol.head(3)
print("\n🏆 Top 3 Categories for Follower Growth:")
for cat, val in cat_fol_top.items():
    print(f"   {cat}: {val:.0f} avg followers gained per post")


## 5. ⚙️ Data Preprocessing

### Feature Selection Strategy
We use only features available **before posting** to simulate real-world prediction:
- Media type, content category, traffic source (categorical → label encoded)
- Caption length, hashtag count (numeric)
- Posting time: hour, day of week, month (temporal)

### Why We Exclude Engagement Metrics as Features
`likes`, `comments`, `reach`, `saves` etc. are only known **after** a post goes live,
so including them would cause **data leakage**.


In [ ]:
feature_cols = ['media_type', 'caption_length', 'hashtags_count',
                'traffic_source', 'content_category', 'Hour', 'Month', 'Day name']
target_col   = 'high_engagement'

df_ml = df[feature_cols + [target_col]].copy()

le = LabelEncoder()
cat_cols = ['media_type', 'traffic_source', 'content_category', 'Day name']
encoders = {}
for col in cat_cols:
    df_ml[col] = le.fit_transform(df_ml[col])
    encoders[col] = dict(zip(le.classes_, le.transform(le.classes_).tolist()))

print("Label Encodings:")
for col, mapping in encoders.items():
    print(f"  {col}: {mapping}")


In [ ]:
X = df_ml[feature_cols]
y = df_ml[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = MinMaxScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training set: {X_train_sc.shape[0]:,} samples")
print(f"Test set:     {X_test_sc.shape[0]:,} samples")
print(f"Features:     {X_train_sc.shape[1]}")
print(f"Class balance (Train): Low={np.bincount(y_train)[0]:,}  High={np.bincount(y_train)[1]:,}")
print(f"Class balance (Test):  Low={np.bincount(y_test)[0]:,}   High={np.bincount(y_test)[1]:,}")


## 6. 🤖 Supervised ML — 9 Classification Models

We test 9 supervised classification algorithms ranging from linear to ensemble methods.
Each model predicts whether a post will achieve **high engagement**.


In [ ]:
models = {
    "Logistic Regression":     LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree":           DecisionTreeClassifier(random_state=42),
    "Random Forest":           RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting":       GradientBoostingClassifier(n_estimators=100, random_state=42),
    "Extra Trees":             ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Support Vector Machine":  CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=42), cv=3),
    "K-Nearest Neighbors":     KNeighborsClassifier(n_neighbors=7),
    "Naive Bayes":             GaussianNB(),
    "AdaBoost":                AdaBoostClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    print(f"  Training {name}...", end=" ")
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    results[name] = {
        "model": model, "y_pred": y_pred, "y_prob": y_prob,
        "accuracy":  accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall":    recall_score(y_test, y_pred),
        "f1":        f1_score(y_test, y_pred),
        "roc_auc":   roc_auc, "fpr": fpr, "tpr": tpr,
    }
    print(f"Acc={results[name]['accuracy']:.4f}  F1={results[name]['f1']:.4f}  AUC={roc_auc:.4f}")

print("\n✅ All models trained.")


## 7. 📊 Model Evaluation

### Metrics Used
| Metric | What It Measures |
|---|---|
| **Accuracy** | Overall correct predictions |
| **Precision** | Of all predicted High — how many were actually High |
| **Recall** | Of all actual High — how many did we catch |
| **F1-Score** | Harmonic mean of Precision and Recall |
| **AUC-ROC** | Model's ability to distinguish classes |


In [ ]:
metrics_df = pd.DataFrame({
    name: {k: v for k, v in res.items() if k in ['accuracy','precision','recall','f1','roc_auc']}
    for name, res in results.items()
}).T.sort_values('f1', ascending=False)

print("\n" + "="*65)
print(f"{'Model':<26} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>7}")
print("-"*65)
for name, row in metrics_df.iterrows():
    print(f"{name:<26} {row['accuracy']:>7.4f} {row['precision']:>7.4f} "
          f"{row['recall']:>7.4f} {row['f1']:>7.4f} {row['roc_auc']:>7.4f}")
print("="*65)


In [ ]:
# Model comparison bar chart
fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor(BG)
x = np.arange(len(metrics_df))
width = 0.17
metric_colors = [ACCENT1, ACCENT4, ACCENT3, ACCENT5, ACCENT2]
metric_names  = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
for i, (met, color, label) in enumerate(zip(metric_names, metric_colors, metric_labels)):
    ax.bar(x + i*width, metrics_df[met], width, label=label, color=color, edgecolor=BG, alpha=0.85)
ax.set_xticks(x + width*2)
ax.set_xticklabels(metrics_df.index, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Score', color=TEXT)
ax.set_title('Model Performance Comparison — All 9 Classifiers', color=TEXT, fontsize=15, fontweight='bold', pad=14)
ax.set_ylim(0, 1.12)
ax.legend(loc='upper right', framealpha=0.2, labelcolor=TEXT, facecolor=SURFACE)
ax.axhline(0.8, color=GRID, linestyle='--', linewidth=1, alpha=0.6)
save_fig("11_model_comparison")


In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor(BG)
for (name, res), color in zip(results.items(), PALETTE):
    ax.plot(res['fpr'], res['tpr'], color=color, linewidth=2,
            label=f"{name} (AUC={res['roc_auc']:.3f})")
ax.plot([0,1],[0,1], color=GRID, linestyle='--', linewidth=1.5, label='Random Classifier')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate', color=TEXT, fontsize=11)
ax.set_ylabel('True Positive Rate', color=TEXT, fontsize=11)
ax.set_title('ROC Curves — All 9 Models', color=TEXT, fontsize=14, fontweight='bold', pad=12)
ax.legend(loc='lower right', framealpha=0.2, labelcolor=TEXT, facecolor=SURFACE, fontsize=9)
save_fig("12_roc_curves")


In [ ]:
# Confusion Matrices — Top 4 models
top4 = metrics_df.head(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.patch.set_facecolor(BG)
fig.suptitle('Confusion Matrices — Top 4 Models', color=TEXT, fontsize=15, fontweight='bold')
for ax, name in zip(axes.flatten(), top4):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                linewidths=1, linecolor=BG, annot_kws={'size': 14, 'color': TEXT, 'fontweight': 'bold'})
    ax.set_title(f"{name}\nF1={results[name]['f1']:.4f}", color=TEXT, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted', color=TEXT); ax.set_ylabel('Actual', color=TEXT)
    ax.set_facecolor(SURFACE)
    ax.set_xticklabels(['Low Eng.', 'High Eng.'], color=TEXT)
    ax.set_yticklabels(['Low Eng.', 'High Eng.'], color=TEXT, rotation=0)
save_fig("13_confusion_matrices")


## 8. 🔧 Hyperparameter Optimization

We apply `RandomizedSearchCV` with 3-fold cross-validation on the best-performing model
to find the optimal configuration.


In [ ]:
best_model_name = metrics_df['f1'].idxmax()
print(f"Best model for optimization: {best_model_name}")

if best_model_name == "Random Forest":
    param_dist = {'n_estimators':[100,200,300],'max_depth':[None,10,20],
                  'min_samples_split':[2,5,10],'min_samples_leaf':[1,2,4]}
    base_model = RandomForestClassifier(random_state=42, n_jobs=-1)
elif best_model_name == "Extra Trees":
    param_dist = {'n_estimators':[100,200,300],'max_depth':[None,10,20],
                  'min_samples_split':[2,5,10],'min_samples_leaf':[1,2,4]}
    base_model = ExtraTreesClassifier(random_state=42, n_jobs=-1)
elif best_model_name == "Gradient Boosting":
    param_dist = {'n_estimators':[100,200],'max_depth':[3,5,7],
                  'learning_rate':[0.05,0.1,0.2],'subsample':[0.7,0.8,1.0]}
    base_model = GradientBoostingClassifier(random_state=42)
else:
    param_dist = {'C':[0.1,1,10],'max_iter':[500,1000]}
    base_model = LogisticRegression(random_state=42)

rs = RandomizedSearchCV(base_model, param_dist, n_iter=10, cv=3, scoring='f1',
                        n_jobs=-1, random_state=42, verbose=0)
rs.fit(X_train_sc, y_train)
best_opt = rs.best_estimator_

y_pred_opt = best_opt.predict(X_test_sc)
y_prob_opt = best_opt.predict_proba(X_test_sc)[:, 1]
fpr_o, tpr_o, _ = roc_curve(y_test, y_prob_opt)
auc_opt = auc(fpr_o, tpr_o)

print(f"Best params: {rs.best_params_}")
print(f"Optimized  — Acc:{accuracy_score(y_test,y_pred_opt):.4f}  "
      f"F1:{f1_score(y_test,y_pred_opt):.4f}  AUC:{auc_opt:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor(BG)
for ax, (y_p, label) in zip(axes, [(results[best_model_name]['y_pred'], 'Base Model'),
                                    (y_pred_opt, 'Optimized Model')]):
    cm = confusion_matrix(y_test, y_p)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                linewidths=1, linecolor=BG, annot_kws={'size':16,'color':TEXT,'fontweight':'bold'})
    ax.set_title(f"{best_model_name}\n{label}", color=TEXT, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted', color=TEXT); ax.set_ylabel('Actual', color=TEXT)
    ax.set_facecolor(SURFACE)
    ax.set_xticklabels(['Low Eng.', 'High Eng.'], color=TEXT)
    ax.set_yticklabels(['Low Eng.', 'High Eng.'], color=TEXT, rotation=0)
fig.suptitle(f'Optimization Effect — {best_model_name}', color=TEXT, fontsize=14, fontweight='bold')
save_fig("15_optimization_comparison")


## 9. 🔍 Feature Importance

In [ ]:
if hasattr(best_opt, 'feature_importances_'):
    fi = pd.Series(best_opt.feature_importances_, index=feature_cols).sort_values(ascending=False)
else:
    from sklearn.inspection import permutation_importance
    r = permutation_importance(best_opt, X_test_sc, y_test, n_repeats=10, random_state=42, n_jobs=-1)
    fi = pd.Series(r.importances_mean, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG)
color_map = [ACCENT1 if v == fi.max() else ACCENT2 if v >= fi.median() else ACCENT5 for v in fi.values]
bars = ax.barh(fi.index[::-1], fi.values[::-1], color=color_map[::-1], edgecolor=BG, linewidth=1.2, height=0.6)
for bar, val in zip(bars, fi.values[::-1]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', color=TEXT, fontsize=10, fontweight='bold')
ax.set_xlabel('Feature Importance Score', color=TEXT, fontsize=11)
ax.set_title(f'Feature Importance — {best_model_name}', color=TEXT, fontsize=14, fontweight='bold', pad=12)
save_fig("16_feature_importance")

print("\nFeature Importance Ranking:")
for feat, score in fi.items():
    print(f"  {feat:<22} {score:.4f}")


## 10. 📈 Regression Analysis — Predicting Engagement Rate

Beyond binary classification, we train regression models to **predict the actual engagement rate value**,
which can help brands estimate performance magnitude, not just high/low categories.


In [ ]:
X_reg = df_ml[feature_cols]
y_reg = df['engagement_rate_calc'].values

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
scaler_r   = MinMaxScaler()
Xr_train_s = scaler_r.fit_transform(Xr_train)
Xr_test_s  = scaler_r.transform(Xr_test)

reg_models = {
    "Ridge Regression":         Ridge(alpha=1.0),
    "Random Forest Regressor":  RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting Reg.":   GradientBoostingRegressor(n_estimators=100, random_state=42),
}

reg_results = {}
for rname, rmodel in reg_models.items():
    print(f"  {rname}...", end=" ")
    rmodel.fit(Xr_train_s, yr_train)
    yr_pred = rmodel.predict(Xr_test_s)
    rmse = float(np.sqrt(mean_squared_error(yr_test, yr_pred)))
    mae  = float(mean_absolute_error(yr_test, yr_pred))
    r2   = float(r2_score(yr_test, yr_pred))
    reg_results[rname] = {"model": rmodel, "rmse": rmse, "mae": mae, "r2": r2, "y_pred": yr_pred}
    print(f"RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor(BG)
names_r   = list(reg_results.keys())
rmse_vals = [reg_results[n]['rmse'] for n in names_r]
r2_vals   = [reg_results[n]['r2']   for n in names_r]

bars = axes[0].bar(names_r, rmse_vals, color=[ACCENT1,ACCENT3,ACCENT5], edgecolor=BG, width=0.5)
for bar, val in zip(bars, rmse_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f'{val:.2f}', ha='center', va='bottom', color=TEXT, fontsize=11, fontweight='bold')
axes[0].set_title('Regression — RMSE (lower = better)', color=TEXT, fontsize=13, fontweight='bold')
axes[0].set_ylabel('RMSE', color=TEXT)
axes[0].tick_params(axis='x', rotation=15)

bars2 = axes[1].bar(names_r, r2_vals, color=[ACCENT1,ACCENT3,ACCENT5], edgecolor=BG, width=0.5)
for bar, val in zip(bars2, r2_vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                 f'{val:.4f}', ha='center', va='bottom', color=TEXT, fontsize=11, fontweight='bold')
axes[1].set_title('Regression — R² Score (higher = better)', color=TEXT, fontsize=13, fontweight='bold')
axes[1].set_ylabel('R² Score', color=TEXT)
axes[1].tick_params(axis='x', rotation=15)
save_fig("18_regression_comparison")


## 11. 💡 Business Insights Dashboard

### ⚠️ Important Note on ML Performance
The classification models achieved **~50% accuracy** (near-random), and regression models
showed **R² ≈ 0**. This is expected and informative:

> **This dataset is synthetically generated** — engagement metrics (likes, reach, impressions) were
> randomly assigned independent of post metadata. In **real-world Instagram data**, post type,
> content category, and timing *do* correlate meaningfully with engagement.

The **EDA-derived insights** (averages per segment) are the primary value here.
The ML pipeline is production-ready and would perform significantly better on real account data.

---


In [ ]:
# Recompute all aggregates for the dashboard
media_eng = df.groupby('media_type')['engagement_rate_calc'].mean().sort_values(ascending=False)
cat_eng   = df.groupby('content_category')['engagement_rate_calc'].mean().sort_values(ascending=False)
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_eng   = df.groupby('Day name')['engagement_rate_calc'].mean().reindex(day_order)
hour_eng  = df.groupby('Hour')['engagement_rate_calc'].mean()
hashtag_bins = pd.cut(df['hashtags_count'],bins=[0,5,10,15,20,30],labels=['1-5','6-10','11-15','16-20','21-30'])
hash_eng  = df.groupby(hashtag_bins)['engagement_rate_calc'].mean()

best_mt   = media_eng.idxmax()
best_cat  = cat_eng.idxmax()
best_day  = day_eng.idxmax()
best_hour = hour_eng.idxmax()
best_hash = str(hash_eng.idxmax())

fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor(BG)
fig.suptitle('Content Strategy Insights — Instagram Performance Guide',
             color=TEXT, fontsize=17, fontweight='bold')
gs = fig.add_gridspec(2, 3, hspace=0.5, wspace=0.4)

panels = [
    (gs[0,0], '🎬 Best Media Type',    best_mt,  f'Avg ER: {media_eng.max():.2f}%', ACCENT1),
    (gs[0,1], '📌 Best Category',      best_cat, f'Avg ER: {cat_eng.max():.2f}%',   ACCENT3),
    (gs[0,2], '📅 Best Day to Post',   best_day, f'Avg ER: {day_eng.max():.2f}%',   ACCENT5),
    (gs[1,0], '⏰ Best Posting Hour',  f'{int(best_hour):02d}:00', f'Avg ER: {hour_eng.max():.2f}%', ACCENT1),
    (gs[1,1], '#️⃣  Best Hashtag Range', f'{best_hash} tags', f'Avg ER: {hash_eng.max():.2f}%', ACCENT3),
]
best_res = sorted(results.items(), key=lambda x: -x[1]['f1'])[0]
panels.append((gs[1,2], '🤖 Best ML Model', best_res[0].replace(' ', '\n'),
               f"F1: {best_res[1]['f1']:.4f}\nAUC: {best_res[1]['roc_auc']:.4f}", ACCENT5))

for spec, title, value, sub, color in panels:
    ax = fig.add_subplot(spec)
    ax.set_facecolor(CARD); ax.axis('off')
    ax.text(0.5,0.78,title,   ha='center',va='center',color=color, fontsize=12,fontweight='bold',transform=ax.transAxes)
    ax.text(0.5,0.45,value,   ha='center',va='center',color=TEXT,  fontsize=16,fontweight='bold',transform=ax.transAxes)
    ax.text(0.5,0.15,sub,     ha='center',va='center',color=ACCENT4,fontsize=10,transform=ax.transAxes)

save_fig("17_business_insights_dashboard")


In [ ]:
print("\n" + "="*60)
print("📋 CONTENT STRATEGY RECOMMENDATIONS")
print("="*60)
print(f"  Media Type:       Post more {best_mt}s for highest engagement")
print(f"  Content Category: {best_cat} content drives the most engagement")
print(f"  Posting Day:      {best_day} is the best day to publish")
print(f"  Posting Hour:     Post at {int(best_hour):02d}:00 for peak reach")
print(f"  Hashtag Strategy: Use {best_hash} hashtags per post")
print()
print("Top 3 Categories by Engagement:")
for cat, val in cat_eng.head(3).items():
    print(f"  {cat:<15} {val:.2f}%")
print()
print("Top 3 Categories by Follower Growth:")
cat_fol = df.groupby('content_category')['followers_gained'].mean().sort_values(ascending=False)
for cat, val in cat_fol.head(3).items():
    print(f"  {cat:<15} {val:.0f} avg followers/post")
print("="*60)


## 12. 💾 Save Model Summary for Web App

In [ ]:
summary = {
    "best_classifier": best_model_name,
    "best_regressor": max(reg_results, key=lambda n: reg_results[n]['r2']),
    "classifier_scores": {
        name: {"accuracy": float(res['accuracy']), "f1": float(res['f1']),
               "roc_auc": float(res['roc_auc']), "precision": float(res['precision']),
               "recall": float(res['recall'])}
        for name, res in results.items()
    },
    "regression_scores": {
        name: {"rmse": r['rmse'], "mae": r['mae'], "r2": r['r2']}
        for name, r in reg_results.items()
    },
    "insights": {
        "best_media_type": str(best_mt),
        "best_category": str(best_cat),
        "best_day": str(best_day),
        "best_hour": int(best_hour),
        "best_hashtag_range": best_hash,
        "best_engagement_rate": float(media_eng.max()),
        "median_engagement_rate": float(df['engagement_rate_calc'].median()),
    },
    "encoders": {k: {str(kk): int(vv) for kk, vv in v.items()} for k, v in encoders.items()},
    "feature_cols": feature_cols,
    "category_avg_engagement": {str(k): float(v) for k, v in cat_eng.items()},
    "media_avg_engagement":    {str(k): float(v) for k, v in media_eng.items()},
    "day_avg_engagement":      {str(k): float(v) for k, v in day_eng.items() if not pd.isna(v)},
    "hour_avg_engagement":     {str(int(k)): float(v) for k, v in hour_eng.items()},
    "category_avg_followers":  {str(k): float(v) for k, v in
                                 df.groupby('content_category')['followers_gained'].mean().items()},
}
with open("model_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("✅ model_summary.json saved.")
print(f"\n🎉 Analysis complete! {len(os.listdir(VIZ_DIR))} visualizations saved to: {VIZ_DIR}/")
